In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.dataset import make_loaders
from src.evaluate import get_logits, predict_loader, summarize
from src.experiment import run_experiment
from src.model import CNNBaseline
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()
paths = {
    'data': '../data/processed',
    'ckpt_dir': '../checkpoints',
    'log_dir': '../logs',
    'results': '../results/experiments.csv',
}

In [ ]:
paths_c = {**paths, 'results': '../results/loss_comparison.csv'}
run_experiment('bce', device, paths_c, epochs=20, loss='bce')

In [ ]:
abl = pd.read_csv('../results/experiments.csv')
bce = pd.read_csv('../results/loss_comparison.csv')
ce_row = abl[abl['experiment'] == 'baseline'].assign(experiment='ce (baseline)')
res = pd.concat([ce_row, bce])
rc = [c for c in res.columns if c.startswith('recall_')]
print(res[['experiment', 'accuracy', 'macro_precision', 'macro_recall', 'macro_f1',
           'lowest_recall_class', 'lowest_precision_class', 'best_epoch', 'gap']].round(3))
print(res[['experiment'] + rc].round(2))

In [ ]:

ce_log = pd.read_csv('../logs/baseline.csv')
bce_log = pd.read_csv('../logs/bce.csv')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, log, t in [(ax[0], ce_log, 'CE loss'), (ax[1], bce_log, 'BCE loss')]:
    a.plot(log['epoch'], log['train_loss'], label='train')
    a.plot(log['epoch'], log['val_loss'], label='val')
    a.set_title(t)
    a.legend()
ax[2].plot(ce_log['epoch'], ce_log['val_macro_f1'], label='CE')
ax[2].plot(bce_log['epoch'], bce_log['val_macro_f1'], label='BCE')
ax[2].set_title('val macro-F1')
ax[2].legend()
plt.tight_layout()
plt.savefig('../results/ce_vs_bce_curves.png', dpi=120)

In [ ]:
_, _, _, val_loader = make_loaders(paths['data'], IMG_SIZE, BATCH, SEED)

stats = {}
for name, kind in [('baseline', 'ce'), ('bce', 'bce')]:
    m = CNNBaseline().to(device)
    m.load_state_dict(torch.load(f'../checkpoints/{name}.pt', weights_only=False)['model_state_dict'])
    logits, _ = get_logits(m, val_loader, device)
    scores = F.softmax(logits, 1) if kind == 'ce' else torch.sigmoid(logits)
    above = (scores > 0.5).sum(1)
    stats[name] = {
        'mean_top_score': scores.max(1).values.mean().item(),
        'mean_sum_of_scores': scores.sum(1).mean().item(),
        'frac_none_above_0.5': (above == 0).float().mean().item(),
        'frac_multiple_above_0.5': (above > 1).float().mean().item(),
    }
print(pd.DataFrame(stats).T.round(3))